# Exploratory Data Analysis
**COEN 330 — Applied Machine Learning**  
**Dataset:** Loan Application Data (45,000 rows, 14 columns)  
**Task:** Binary classification — predict `is_risky` (1 = Risky/Deny, 0 = Safe/Approve)

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None) # display all columns    
sns.set_theme(style='whitegrid', palette='muted') # set the theme and palette
plt.rcParams['figure.dpi'] = 120 # set the figure dpi

# random seed for reproducibility
SEED = 42

## 1. Load Data & Create Target

In [9]:
df = pd.read_csv('../data/raw/dataset.csv')

# Create target: is_risky = 1 - loan_status
# loan_status: 1=Approved, 0=Denied → is_risky: 1=Risky, 0=Safe
df['is_risky'] = 1 - df['loan_status']

print(f'Shape: {df.shape}')
df.head() # first 5 rows of the dataframe

FileNotFoundError: [Errno 2] No such file or directory: '../dataset.csv'

## 2. Basic Dataset Info

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  str    
 2   person_education                45000 non-null  str    
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  str    
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  str    
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  45000 non-n

In [6]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
person_age,45000.0,NaN,NaN,NaN,27.764178,6.045108,20.0,24.0,26.0,30.0,144.0
person_gender,45000,2,male,24841,NaN,NaN,NaN,NaN,NaN,NaN,NaN
person_education,45000,5,Bachelor,13399,NaN,NaN,NaN,NaN,NaN,NaN,NaN
person_income,45000.0,NaN,NaN,NaN,80319.053222,80422.498632,8000.0,47204.0,67048.0,95789.25,7200766.0
person_emp_exp,45000.0,NaN,NaN,NaN,5.410333,6.063532,0.0,1.0,4.0,8.0,125.0
person_home_ownership,45000,4,RENT,23443,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_amnt,45000.0,NaN,NaN,NaN,9583.157556,6314.886691,500.0,5000.0,8000.0,12237.25,35000.0
loan_intent,45000,6,EDUCATION,9153,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_int_rate,45000.0,NaN,NaN,NaN,11.006606,2.978808,5.42,8.59,11.01,12.99,20.0
loan_percent_income,45000.0,NaN,NaN,NaN,0.139725,0.087212,0.0,0.07,0.12,0.19,0.66


## 3. Missing Values

In [7]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found.')

Missing values per column:
No missing values found.


## 4. Target Distribution

> **Note:** In this dataset, `is_risky=1` (Risky) is the **majority class** (~78%), which is the opposite of typical risk datasets. This must be accounted for in metric selection and model evaluation.

In [ ]:
target_counts = df['is_risky'].value_counts()
target_pct = df['is_risky'].value_counts(normalize=True) * 100

print('Class distribution:')
print(pd.DataFrame({'Count': target_counts, 'Percent': target_pct.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Safe (0)', 'Risky (1)'], target_counts.sort_index(),
              color=['steelblue', 'tomato'], edgecolor='white')
for bar, pct in zip(bars, target_pct.sort_index()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'{pct:.1f}%', ha='center', fontsize=11)
ax.set_title('Target Class Distribution (is_risky)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../results/plots/target_distribution.png')
plt.show()

## 5. Numerical Feature Distributions

In [ ]:
num_cols = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
            'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Numerical Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/plots/numerical_distributions.png')
plt.show()

### 5.1 Outlier Analysis — `person_age`

Maximum age of 144 is biologically impossible — this is a data quality issue.

In [ ]:
print('person_age statistics:')
print(df['person_age'].describe())
print(f"\nRows with age > 80: {(df['person_age'] > 80).sum()}")
print(f"Rows with age > 100: {(df['person_age'] > 100).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['person_age'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('person_age — Full Range')
sns.boxplot(x=df['person_age'], ax=axes[1], color='steelblue')
axes[1].set_title('person_age — Boxplot')
plt.tight_layout()
plt.savefig('../results/plots/person_age_outliers.png')
plt.show()

### 5.2 Skewness Analysis — `person_income`

`person_income` is heavily right-skewed. A log-transform or capping strategy will be evaluated in the preprocessing notebook.

In [ ]:
print('person_income statistics:')
print(df['person_income'].describe())
print(f"\nSkewness: {df['person_income'].skew():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['person_income'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('person_income — Original')
sns.histplot(np.log1p(df['person_income']), kde=True, ax=axes[1], color='mediumseagreen')
axes[1].set_title('person_income — log1p Transformed')
plt.tight_layout()
plt.savefig('../results/plots/person_income_skewness.png')
plt.show()

## 6. Categorical Feature Distributions

In [ ]:
cat_cols = ['person_gender', 'person_education', 'person_home_ownership',
            'loan_intent', 'previous_loan_defaults_on_file']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flatten(), cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, palette='muted')
    ax.set_title(col)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
axes.flatten()[-1].set_visible(False)
plt.suptitle('Categorical Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/plots/categorical_distributions.png')
plt.show()

## 7. Near-Perfect Predictor — `previous_loan_defaults_on_file`

> **Important finding:** `previous_loan_defaults_on_file = Yes` maps nearly perfectly to `is_risky = 1`. This feature may make the problem easier than it would be in a realistic setting and must be discussed in the report.

In [ ]:
cross = pd.crosstab(df['previous_loan_defaults_on_file'], df['is_risky'],
                    normalize='index') * 100
print('is_risky rate by previous_loan_defaults_on_file (%):')
print(cross.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
cross.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='white')
ax.set_title('is_risky Rate by previous_loan_defaults_on_file')
ax.set_ylabel('Percentage (%)')
ax.set_xlabel('previous_loan_defaults_on_file')
ax.legend(['Safe (0)', 'Risky (1)'])
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../results/plots/defaults_vs_risky.png')
plt.show()

## 8. Feature vs. Target Relationships

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(data=df, x='is_risky', y=col, ax=ax,
                palette={0: 'steelblue', 1: 'tomato'})
    ax.set_title(col)
    ax.set_xlabel('is_risky')
plt.suptitle('Numerical Features vs. Target (is_risky)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/plots/numerical_vs_target.png')
plt.show()

## 9. Correlation Heatmap

In [ ]:
corr_cols = num_cols + ['is_risky']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap (Numerical Features + Target)')
plt.tight_layout()
plt.savefig('../results/plots/correlation_heatmap.png')
plt.show()

## 10. EDA Summary

| Finding | Detail | Action |
|---|---|---|
| No missing values | All 14 columns complete | No imputation needed |
| Class imbalance (inverted) | ~78% Risky, ~22% Safe | Use Recall as primary metric; discuss in report |
| `person_age` outliers | Max = 144 (invalid) | Cap at 80 in preprocessing |
| `person_income` skewness | Heavily right-skewed | Apply log1p transform |
| `previous_loan_defaults_on_file` | Near-perfect predictor of is_risky | Flag in report as potential shortcut feature |
| `person_education` | Ordinal (5 levels) | Use ordinal encoding |
| `loan_status` | Inverse of target | **Drop before training** (data leakage) |